In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Import Statsmodels
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.tools.eval_measures import rmse, aic

In [11]:
#df_vital = pd.read_csv('vitals_sbp_interpolated.csv')
df_vitals = pd.read_csv('vitals2.csv')
df_beta = pd.read_csv('beta-blocker-dosages.csv')

In [12]:
df_vitals['timestamp'] = pd.to_datetime(df_vitals['timestamp'], errors='coerce')

# might have to add or remove columns due to availability of the data
df_vitals = df_vitals[['patient_uuid', 'timestamp', 'HR', 'SBP']]



df_vitals = df_vitals[(df_vitals.timestamp > '2018-10-01') & (df_vitals.timestamp < '2020-01-14')]
df_vitals = df_vitals[~df_vitals['SBP'].isna()]
df_vitals = df_vitals[~df_vitals['HR'].isna()]

#df_beta['timestamp'] = pd.to_datetime(df_beta['timestamp'], errors='coerce')
#df_beta = df_beta[~df_beta['timestamp'].isna()]
#df_beta = df_beta[df_beta['status'] == 'active']

# list of unique patients on beta-blocker
beta_patients = df_beta['patient_uuid'].unique()

# vitals dataset subset containing patients in beta blockers
df_vitals2 = df_vitals[df_vitals['patient_uuid'].isin(beta_patients)]

# columns with number of non-empty rows
(~df_vitals2.isna()).sum() / len(df_vitals2) * 100

patient_uuid    100.0
timestamp       100.0
HR              100.0
SBP             100.0
dtype: float64

In [13]:
df_beta.head()

,Unnamed: 0,timezoneOffset,medication,eventTypeAndTimestamp,patient_uuid,timestamp,updated,created
0,0,"{'_type': 'decimal', 'value': '-240'}","{'timeTaken': ['8:00 AM'], 'dosage': '50mg', '...",MED-1565907407895,88748003,"{'_type': 'decimal', 'value': '1565907407895'}",NaN,NaN
1,1,"{'_type': 'decimal', 'value': '-240'}","{'timeTaken': ['8:00 AM'], 'dosage': '25mg', '...",MED-1568314798436,12742003,"{'_type': 'decimal', 'value': '1568314798436'}",NaN,NaN
2,2,"{'_type': 'decimal', 'value': '-240'}","{'timeTaken': ['8:00 AM'], 'dosage': '25mg', '...",MED-1571411967818,1386626,"{'_type': 'decimal', 'value': '1571411967818'}","{'_type': 'decimal', 'value': '1571411967488'}","{'_type': 'decimal', 'value': '1571411967488'}"
3,3,"{'_type': 'decimal', 'value': '0'}","{'dosage': '50mg', 'timeTaken': [], 'genericNa...",MED-1549299583000,6719896,"{'_type': 'decimal', 'value': '1549299583000'}",NaN,NaN
4,4,"{'_type': 'decimal', 'value': '0'}","{'dosage': '25mg', 'timeTaken': [], 'genericNa...",MED-1550616142000,48878864,"{'_type': 'decimal', 'value': '1550616142000'}",NaN,NaN


In [14]:
df_vitals2.head()

,patient_uuid,timestamp,HR,SBP
823,1386626,2019-07-23 12:31:00,77.0,99.0
884,6719896,2019-01-02 09:46:00,80.0,160.0
885,6719896,2019-01-03 09:01:00,80.0,160.0
886,6719896,2019-01-03 09:02:00,79.0,162.0
887,6719896,2019-01-03 09:42:00,80.0,159.0


In [16]:
df_daily = df_vitals.groupby([df_vitals.patient_uuid, pd.Grouper(key='timestamp', freq='D')]).mean().reset_index()
df_daily2 = df_vitals2.groupby([df_vitals2.patient_uuid, pd.Grouper(key='timestamp', freq='D')]).mean().reset_index()

In [20]:
df_recordings_per_patient = df_daily.groupby('patient_uuid').count().reset_index()
patients_over_30 = list(df_recordings_per_patient[df_recordings_per_patient.HR >= 30].patient_uuid.unique())
df_daily = df_daily[df_daily.patient_uuid.isin(patients_over_30)]

df_recordings_per_patient2 = df_daily.groupby('patient_uuid').count().reset_index()
patients_over_30_2 = list(df_recordings_per_patient2[df_recordings_per_patient2.HR >= 30].patient_uuid.unique())
df_daily2 = df_daily2[df_daily2.patient_uuid.isin(patients_over_30_2)]

In [25]:
df_recordings_per_patient2[df_recordings_per_patient2.HR > 450]

,patient_uuid,timestamp,HR,SBP
334,36279955,463,463,463


In [133]:
df_temp1 = df_daily[df_daily.patient_uuid == 36279955]
df_temp1.index = pd.DatetimeIndex(df_temp1.timestamp).to_period('D')
df_temp1 = df_temp1[['HR','SBP']].copy()
df_temp1.head()

,HR,SBP
timestamp,,
2018-10-01,73.333333,148.666667
2018-10-02,80.000000,168.000000
2018-10-03,80.000000,176.000000
2018-10-04,74.000000,179.000000
2018-10-05,72.000000,135.000000


In [19]:
from statsmodels.tsa.stattools import grangercausalitytests
maxlag=12
test = 'ssr_chi2test'
def grangers_causation_matrix(data, variables, test='ssr_chi2test', verbose=False):    
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r, c]], maxlag=maxlag, verbose=False)
            p_values = [round(test_result[i+1][0][test][1],4) for i in range(maxlag)]
            if verbose: print(f'Y = {r}, X = {c}, P Values = {p_values}')
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    df.columns = [var + '_x' for var in variables]
    df.index = [var + '_y' for var in variables]
    return df



In [34]:
grangers_causation_matrix(df_temp1, variables = df_temp1.columns)

,HR_x,SBP_x
HR_y,1.0000,0.0003
SBP_y,0.0118,1.0000


In [35]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

def cointegration_test(df, alpha=0.05): 
    """Perform Johanson's Cointegration Test and Report Summary"""
    out = coint_johansen(df,-1,5)
    d = {'0.90':0, '0.95':1, '0.99':2}
    traces = out.lr1
    cvts = out.cvt[:, d[str(1-alpha)]]
    def adjust(val, length= 6): return str(val).ljust(length)

    # Summary
    print('Name   ::  Test Stat > C(95%)    =>   Signif  \n', '--'*20)
    for col, trace, cvt in zip(df.columns, traces, cvts):
        print(adjust(col), ':: ', adjust(round(trace,2), 9), ">", adjust(cvt, 8), ' =>  ' , trace > cvt)



In [36]:
cointegration_test(df_temp1)

Name   ::  Test Stat > C(95%)    =>   Signif  
 ----------------------------------------
HR     ::  17.38     > 12.3212   =>   True
SBP    ::  0.13      > 4.1296    =>   False


In [92]:
nobs = 7
df_train, df_test = df_temp1[0:-nobs], df_temp1[-nobs:]

In [42]:
def adfuller_test(series, signif=0.05, name='', verbose=False):
    """Perform ADFuller to test for Stationarity of given series and print report"""
    r = adfuller(series, autolag='AIC')
    p_value = round(r[1], 4)

    if p_value <= signif:
        is_stationary = True
    else:
        is_stationary = False
        
    return is_stationary

In [44]:
# ADF Test on each column
for name, column in df_train.iteritems():
    print(adfuller_test(column, name=column.name))
    

False
True


In [48]:
df_differenced = df_train.diff().dropna()
df_differenced.head()

,HR,SBP
timestamp,,
2018-10-02,6.666667,19.333333
2018-10-03,0.000000,8.000000
2018-10-04,-6.000000,3.000000
2018-10-05,-2.000000,-44.000000
2018-10-06,8.000000,25.000000


In [46]:
# ADF Test on each column
for name, column in df_differenced.iteritems():
    print(adfuller_test(column, name=column.name))

True
True


In [83]:
model = VAR(df_differenced)
x = model.select_order(maxlags=20)
lag_order = x.aic
model_fitted = model.fit(lag_order)
model_fitted.summary()

/home/ablew/anaconda3/lib/python3.7/site-packages/statsmodels/tsa/base/tsa_model.py:225: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  ' ignored when e.g. forecasting.', ValueWarning)


  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sun, 01, Mar, 2020
Time:                     07:38:08
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                    8.95959
Nobs:                     446.000    HQIC:                   8.74798
Log likelihood:          -3147.78    FPE:                    5488.10
AIC:                      8.61023    Det(Omega_mle):         5048.77
--------------------------------------------------------------------
Results for equation HR
            coefficient       std. error           t-stat            prob
-------------------------------------------------------------------------
const         -0.045679         0.270099           -0.169           0.866
L1.HR         -0.739556         0.049227          -15.023           0.000
L1.SBP        -0.020494         0.021909           -0.935           0.350
L2.HR         -0.

In [56]:
from statsmodels.stats.stattools import durbin_watson
out = durbin_watson(model_fitted.resid)

for col, val in zip(df_temp1.columns, out):
    print(col, ':', round(val, 2))

HR : 2.0
SBP : 1.99


In [57]:
# Input data for forecasting
forecast_input = df_differenced.values[-lag_order:]
forecast_input

9


array([[-12.16666667,   4.16666667],
       [ 10.5       ,   7.5       ],
       [  0.        ,   3.5       ],
       [ -9.        ,  -5.5       ],
       [  1.33333333,  -1.        ],
       [  9.66666667,   9.5       ],
       [-10.66666667,  -8.83333333],
       [ 10.16666667,   0.33333333],
       [ -6.5       ,  -9.        ]])

In [61]:
# Forecast
fc = model_fitted.forecast(y=forecast_input, steps=nobs)
df_forecast = pd.DataFrame(fc, index=df_temp1.index[-nobs:], columns=df_temp1.columns + '_forecast')
df_forecast

,HR_forecast,SBP_forecast
timestamp,,
2020-01-07,0.812596,6.738867
2020-01-08,1.329290,-1.584217
2020-01-09,-0.403128,2.782218
2020-01-10,-0.463273,-0.242434
2020-01-11,0.502073,-1.089267
2020-01-12,-0.605055,0.498256
2020-01-13,0.841037,0.715500


In [62]:
def invert_transformation(df_train, df_forecast, second_diff=False):
    """Revert back the differencing to get the forecast to original scale."""
    df_fc = df_forecast.copy()
    columns = df_train.columns
    for col in columns:        
        # Roll back 2nd Diff
        if second_diff:
            df_fc[str(col)+'_forecast'] = (df_train[col].iloc[-1]-df_train[col].iloc[-2]) + df_fc[str(col)+'_forecast'].cumsum()
        # Roll back 1st Diff
        df_fc[str(col)+'_forecast'] = df_train[col].iloc[-1] + df_fc[str(col)+'_forecast'].cumsum()
    return df_fc

In [70]:
df_forecast = invert_transformation(df_train, df_forecast, second_diff=False)

In [71]:
df_forecast

,HR_forecast,SBP_forecast
timestamp,,
2020-01-07,70.812596,132.738867
2020-01-08,72.141885,131.154651
2020-01-09,71.738757,133.936868
2020-01-10,71.275484,133.694435
2020-01-11,71.777557,132.605168
2020-01-12,71.172502,133.103424
2020-01-13,72.013539,133.818924


In [66]:
df_test

,HR,SBP
timestamp,,
2020-01-07,80.666667,131.0
2020-01-08,85.500000,130.0
2020-01-09,71.500000,142.5
2020-01-10,67.000000,167.0
2020-01-11,78.000000,128.0
2020-01-12,82.500000,153.5
2020-01-13,72.666667,120.0


In [67]:
def forecast_accuracy(forecast, actual):
    forecast = forecast.values
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))  # MAPE
    me = np.mean(forecast - actual)             # ME
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return mape, me, rmse

In [68]:
forecast_accuracy(df_results, df_test)

(HR     0.081659
 SBP    0.080814
 dtype: float64, HR    -5.271573
 SBP   -5.849666
 dtype: float64, HR      8.118201
 SBP    16.102951
 dtype: float64)

In [151]:
def var_model(patient):
    nobs = 7
    diff = 0
    max_diff = 2
    
    df_patient = df_daily[df_daily.patient_uuid == patient]
    df_patient.index = pd.DatetimeIndex(df_patient.timestamp).to_period('D')
    df_patient = df_patient[['HR','SBP']].copy()
    
    df_train, df_test = df_patient[0:-nobs], df_patient[-nobs:]
    df_train2 = df_train.copy()
    
    """
    are_stationary = []
    for name, column in df_train.iteritems():
        are_stationary.append(adfuller_test(column, name=column.name))
        
    while (all(are_stationary) == False) and (diff < max_diff):
        are_stationary = []
        df_train = df_train.diff().dropna()
        for name, column in df_train.iteritems():
            are_stationary.append(adfuller_test(column, name=column.name))

        diff += 1
    """
    
    model = VAR(df_train)
    #x = model.select_order(maxlags=12) # when gap fixed
    lag_order = 7 # x.aic when gap fixed
    model_fitted = model.fit(lag_order)
    
    forecast_input = df_train.values[-lag_order:]
    
    fc = model_fitted.forecast(y=forecast_input, steps=nobs)
    df_forecast = pd.DataFrame(fc, index=df_temp1.index[-nobs:], columns=df_temp1.columns + '_forecast')
    
    if diff == 1:
        df_forecast = invert_transformation(df_train2, df_forecast, second_diff=False)
    elif diff == 2:
        df_forecast = invert_transformation(df_train2, df_forecast, second_diff=True)
        
    return df_forecast, df_train2, df_test
        
        
        

In [140]:
a = forecast_accuracy(var_model(36279955)[0], var_model(36279955)[2])
a

(HR     0.080622
 SBP    0.088388
 dtype: float64, HR    -3.831972
 SBP   -4.758819
 dtype: float64, HR      7.532959
 SBP    16.493999
 dtype: float64)

In [146]:
var_model(patients_over_30[7]) # patient with gap

ValueError: array must not contain infs or NaNs

In [150]:
df_daily[df_daily.patient_uuid == patients_over_30[7]].tail(10)

,patient_uuid,timestamp,HR,SBP
641,2075791,2019-11-28,50.500000,132.500000
642,2075791,2019-11-29,49.000000,137.000000
643,2075791,2019-11-30,49.500000,128.000000
644,2075791,2019-12-02,55.000000,127.000000
645,2075791,2019-12-03,53.000000,124.000000
646,2075791,2019-12-04,53.000000,126.000000
647,2075791,2019-12-05,56.500000,130.500000
648,2075791,2019-12-06,50.666667,122.666667
649,2075791,2019-12-07,52.500000,132.500000
650,2075791,2019-12-08,51.000000,127.000000


In [152]:
HR_mape = []
HR_me = []
HR_rsme = []
SBP_mape = []
SBP_me = []
SBP_rsme = []

for patient in patients_over_30:
    results_forecast, results_train, results_test = var_model(patient)
    accuracy_indicators = forecast_accuracy(results_forecast, results_test)
    HR_mape.append(accuracy_indicators[0][0])
    HR_me.append(accuracy_indicators[1][0])
    HR_rsme.append(accuracy_indicators[2][0])
    SBP_mape.append(accuracy_indicators[0][1])
    SBP_me.append(accuracy_indicators[1][1])
    SBP_rsme.append(accuracy_indicators[2][1])

In [127]:
df_model = pd.DataFrame(columns = ['patient_uuid', 'HR_mape', 'HR_me', 'HR_rsme', 'SBP_mape', 'SBP_me', 'SBP_rsme'])
df_model.patient_uuid = patients_over_30
df_model.HR_mape = HR_mape
df_model.HR_me = HR_me
df_model.HR_rsme = HR_rsme
df_model.SBP_mape = SBP_mape
df_model.SBP_me = SBP_me
df_model.SBP_rsme = SBP_rsme

df_model.head()

,patient_uuid,HR_mape,HR_me,HR_rsme,SBP_mape,SBP_me,SBP_rsme
0,1048697,0.092742,3.149454,9.160463,0.105386,0.874800,15.511969
1,1127912,0.108441,-3.464069,12.212388,0.049998,-6.112574,7.274438
2,1164638,0.108912,6.050980,10.334925,0.063233,-4.347291,11.058932
3,1166047,0.060921,-4.451583,5.735058,0.119151,7.929419,15.062484
4,1193821,0.065590,-0.629971,4.904616,0.096961,5.668330,12.730098
